# Custom GPT-Style Base Model Pretraining

Upload only these files to Colab:

```text
colab_pretrain_base_model.ipynb
hf_tokenizer.zip
```

`hf_tokenizer.zip` should contain your custom Hugging Face tokenizer files from `tokenizer/hf_tokenizer/`:

```text
tokenizer.json
tokenizer_config.json
special_tokens_map.json
```

WikiText is downloaded inside Colab, so you do not need to upload the WikiText dataset folder.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
%pip install -q 'tokenizers>=0.15.0' 'transformers>=4.38.0' 'torch>=2.0.0' 'accelerate>=0.26.0' 'datasets>=2.18.0'

## 3. Upload Tokenizer Zip

When prompted, upload `hf_tokenizer.zip`.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import zipfile

WORK_DIR = Path('/content/custom_bpe_base')
TOKENIZER_DIR = WORK_DIR / 'tokenizer' / 'hf_tokenizer'
OUTPUT_DIR = WORK_DIR / 'model' / 'base_model'

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
zip_candidates = [Path(name) for name in uploaded if name.endswith('.zip')]
if not zip_candidates:
    raise FileNotFoundError('Upload hf_tokenizer.zip before continuing.')

with zipfile.ZipFile(zip_candidates[0]) as zip_file:
    zip_file.extractall(WORK_DIR / 'uploaded_tokenizer')

tokenizer_jsons = list((WORK_DIR / 'uploaded_tokenizer').rglob('tokenizer.json'))
if not tokenizer_jsons:
    raise FileNotFoundError('The zip must contain tokenizer.json from tokenizer/hf_tokenizer/.')

source_tokenizer_dir = tokenizer_jsons[0].parent
for file_path in source_tokenizer_dir.iterdir():
    if file_path.is_file():
        shutil.copy2(file_path, TOKENIZER_DIR / file_path.name)

required = ['tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json']
missing = [name for name in required if not (TOKENIZER_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Missing tokenizer files: ' + ', '.join(missing))

print(f'Tokenizer ready at: {TOKENIZER_DIR}')

## 4. Training Settings

Run once with `SMOKE_TEST = True` if you want a quick setup check. For actual base training, keep it `False`.

In [ ]:
SMOKE_TEST = False

# T4-oriented defaults. These intentionally do more work per optimizer step than
# the local script so Colab's 15 GB GPU is not mostly idle.
BLOCK_SIZE = 512
MAX_TRAIN_LINES = 1_000_000
MAX_VALIDATION_LINES = 10_000
NUM_TRAIN_EPOCHS = 1
BATCH_SIZE = 96
GRADIENT_ACCUMULATION_STEPS = 1
DATASET_NUM_PROC = 2
DATALOADER_NUM_WORKERS = 2
DATALOADER_PREFETCH_FACTOR = 4
AUTO_FIND_BATCH_SIZE = True

# Larger than the local 6-layer/256-hidden model, but still practical on a free T4.
MODEL_EMBED_DIM = 512
MODEL_LAYERS = 8
MODEL_HEADS = 8
MODEL_INNER_DIM = 2048

if SMOKE_TEST:
    BLOCK_SIZE = 256
    MAX_TRAIN_LINES = 512
    MAX_VALIDATION_LINES = 128
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 1
    DATASET_NUM_PROC = 1
    DATALOADER_NUM_WORKERS = 0
    DATALOADER_PREFETCH_FACTOR = 2
    AUTO_FIND_BATCH_SIZE = False
    MODEL_EMBED_DIM = 256
    MODEL_LAYERS = 6
    MODEL_HEADS = 4
    MODEL_INNER_DIM = 1024

print({
    'SMOKE_TEST': SMOKE_TEST,
    'BLOCK_SIZE': BLOCK_SIZE,
    'MAX_TRAIN_LINES': MAX_TRAIN_LINES,
    'MAX_VALIDATION_LINES': MAX_VALIDATION_LINES,
    'NUM_TRAIN_EPOCHS': NUM_TRAIN_EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'GRADIENT_ACCUMULATION_STEPS': GRADIENT_ACCUMULATION_STEPS,
    'DATASET_NUM_PROC': DATASET_NUM_PROC,
    'DATALOADER_NUM_WORKERS': DATALOADER_NUM_WORKERS,
    'DATALOADER_PREFETCH_FACTOR': DATALOADER_PREFETCH_FACTOR,
    'AUTO_FIND_BATCH_SIZE': AUTO_FIND_BATCH_SIZE,
    'MODEL_EMBED_DIM': MODEL_EMBED_DIM,
    'MODEL_LAYERS': MODEL_LAYERS,
    'MODEL_HEADS': MODEL_HEADS,
    'MODEL_INNER_DIM': MODEL_INNER_DIM,
})

## 5. Define Training Code

In [ ]:
import inspect
import os
import re
import subprocess
from pathlib import Path

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('HF_HOME', str(WORK_DIR / 'hf_home'))
os.environ.setdefault('HF_DATASETS_CACHE', str(WORK_DIR / 'hf_datasets_cache'))
os.environ.setdefault('TMPDIR', str(WORK_DIR / 'tmp'))
Path(os.environ['TMPDIR']).mkdir(parents=True, exist_ok=True)

import torch
from datasets import Dataset, load_dataset
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    default_data_collator,
)

WHITESPACE_RE = re.compile(r'\s+')

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')


def clean_wikitext_line(text: str) -> str | None:
    text = text.replace('—', ',').replace('–', '-')
    text = WHITESPACE_RE.sub(' ', text).strip()
    if len(text) < 40:
        return None
    if text.startswith('=') and text.endswith('='):
        return None
    return text


def load_tokenizer() -> PreTrainedTokenizerFast:
    tokenizer = PreTrainedTokenizerFast.from_pretrained(str(TOKENIZER_DIR))
    if tokenizer.pad_token_id is None or tokenizer.eos_token_id is None:
        raise ValueError('Tokenizer must define both pad_token_id and eos_token_id.')
    tokenizer.model_max_length = BLOCK_SIZE
    return tokenizer


def select_rows(dataset: Dataset, max_rows: int) -> Dataset:
    if len(dataset) <= max_rows:
        return dataset
    return dataset.select(range(max_rows))


def load_wikitext_split(split: str) -> Dataset:
    try:
        return load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split=split)
    except Exception as error:
        raise RuntimeError(
            'Could not download WikiText from Hugging Face. Check Colab internet access, then rerun this cell.'
        ) from error


def prepare_lm_dataset(split: str, tokenizer: PreTrainedTokenizerFast, max_lines: int) -> Dataset:
    dataset = select_rows(load_wikitext_split(split), max_lines)
    map_workers = {'num_proc': DATASET_NUM_PROC} if DATASET_NUM_PROC > 1 else {}

    def clean_batch(batch: dict[str, list[str]]) -> dict[str, list[str]]:
        cleaned_texts = []
        for text in batch['text']:
            cleaned = clean_wikitext_line(text)
            if cleaned is not None:
                cleaned_texts.append(cleaned)
        return {'text': cleaned_texts}

    def tokenize_batch(batch: dict[str, list[str]]) -> dict[str, list[list[int]]]:
        return tokenizer(batch['text'], add_special_tokens=False, verbose=False)

    def group_texts(batch: dict[str, list[list[int]]]) -> dict[str, list[list[int]]]:
        token_ids = []
        for input_ids in batch['input_ids']:
            token_ids.extend(input_ids)
            token_ids.append(tokenizer.eos_token_id)

        total_length = len(token_ids) // BLOCK_SIZE * BLOCK_SIZE
        token_ids = token_ids[:total_length]
        chunks = [token_ids[index:index + BLOCK_SIZE] for index in range(0, total_length, BLOCK_SIZE)]
        return {
            'input_ids': chunks,
            'attention_mask': [[1] * BLOCK_SIZE for _ in chunks],
            'labels': [chunk.copy() for chunk in chunks],
        }

    cleaned = dataset.map(
        clean_batch,
        batched=True,
        remove_columns=dataset.column_names,
        desc=f'Cleaning WikiText {split}',
        **map_workers,
    )
    tokenized = cleaned.map(
        tokenize_batch,
        batched=True,
        remove_columns=cleaned.column_names,
        desc=f'Tokenizing WikiText {split}',
        **map_workers,
    )
    grouped = tokenized.map(
        group_texts,
        batched=True,
        batch_size=2_000,
        remove_columns=tokenized.column_names,
        desc=f'Packing WikiText {split}',
        **map_workers,
    )
    if len(grouped) == 0:
        raise ValueError(f'No usable language-model chunks found for split: {split}')
    return grouped.with_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


def build_model(tokenizer: PreTrainedTokenizerFast) -> GPT2LMHeadModel:
    config = GPT2Config(
        vocab_size=len(tokenizer),
        n_positions=BLOCK_SIZE,
        n_ctx=BLOCK_SIZE,
        n_embd=MODEL_EMBED_DIM,
        n_layer=MODEL_LAYERS,
        n_head=MODEL_HEADS,
        n_inner=MODEL_INNER_DIM,
        activation_function='gelu_new',
        resid_pdrop=0.1,
        embd_pdrop=0.1,
        attn_pdrop=0.1,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    model = GPT2LMHeadModel(config)
    model.config.use_cache = False
    return model


def precision_flags() -> tuple[bool, bool]:
    if not torch.cuda.is_available():
        return False, False
    if torch.cuda.is_bf16_supported():
        return False, True
    return True, False


def supported_training_args(args: dict) -> dict:
    supported_args = inspect.signature(TrainingArguments.__init__).parameters
    return {key: value for key, value in args.items() if key in supported_args and value is not None}


def build_training_args() -> TrainingArguments:
    supported_args = inspect.signature(TrainingArguments.__init__).parameters
    strategy_name = 'eval_strategy' if 'eval_strategy' in supported_args else 'evaluation_strategy'
    fp16, bf16 = precision_flags()
    args = {
        'output_dir': str(OUTPUT_DIR),
        'overwrite_output_dir': True,
        'num_train_epochs': NUM_TRAIN_EPOCHS,
        'per_device_train_batch_size': BATCH_SIZE,
        'per_device_eval_batch_size': min(BATCH_SIZE, 64),
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'learning_rate': 5e-4,
        'weight_decay': 0.01,
        'warmup_ratio': 0.03,
        'lr_scheduler_type': 'cosine',
        strategy_name: 'steps',
        'eval_steps': 2_000,
        'save_steps': 2_000,
        'logging_steps': 50,
        'fp16': fp16,
        'bf16': bf16,
        'report_to': [],
        'save_total_limit': 2,
        'dataloader_num_workers': DATALOADER_NUM_WORKERS,
        'dataloader_pin_memory': torch.cuda.is_available(),
        'dataloader_persistent_workers': DATALOADER_NUM_WORKERS > 0,
        'dataloader_prefetch_factor': DATALOADER_PREFETCH_FACTOR if DATALOADER_NUM_WORKERS > 0 else None,
        'auto_find_batch_size': AUTO_FIND_BATCH_SIZE,
        'include_tokens_per_second': True,
        'seed': 42,
        'optim': 'adamw_torch_fused' if torch.cuda.is_available() else 'adamw_torch',
    }
    return TrainingArguments(**supported_training_args(args))


def tokenizer_trainer_arg(tokenizer: PreTrainedTokenizerFast) -> dict[str, PreTrainedTokenizerFast]:
    supported_args = inspect.signature(Trainer.__init__).parameters
    if 'processing_class' in supported_args:
        return {'processing_class': tokenizer}
    if 'tokenizer' in supported_args:
        return {'tokenizer': tokenizer}
    return {}


def model_parameter_count(model: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


def print_runtime_summary(model: torch.nn.Module, train_dataset: Dataset, validation_dataset: Dataset) -> None:
    print(f'Torch: {torch.__version__}')
    print(f'Model parameters: {model_parameter_count(model):,}')
    print(f'Train chunks: {len(train_dataset):,}; validation chunks: {len(validation_dataset):,}')
    print(f'Tokens per optimizer step: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * BLOCK_SIZE:,}')
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
        fp16, bf16 = precision_flags()
        precision = 'bf16' if bf16 else 'fp16' if fp16 else 'fp32'
        print(f'CUDA device: {device_name} ({total_gib:.1f} GiB), precision: {precision}')
        subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,memory.total,memory.used,utilization.gpu',
            '--format=csv,noheader,nounits',
        ], check=False)
    else:
        print('CUDA is not available; switch Colab runtime to GPU before training.')


class CudaMemoryCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.cuda.is_available():
            allocated = torch.cuda.max_memory_allocated() / 1024**3
            reserved = torch.cuda.max_memory_reserved() / 1024**3
            print(f'CUDA peak memory: allocated={allocated:.2f} GiB reserved={reserved:.2f} GiB')


## 6. Train Base Model

In [ ]:
tokenizer = load_tokenizer()
model = build_model(tokenizer)

train_dataset = prepare_lm_dataset('train', tokenizer, MAX_TRAIN_LINES)
validation_dataset = prepare_lm_dataset('validation', tokenizer, MAX_VALIDATION_LINES)
training_args = build_training_args()
trainer_tokenizer_kwargs = tokenizer_trainer_arg(tokenizer)
print_runtime_summary(model, train_dataset, validation_dataset)
print(f"Trainer tokenizer argument: {next(iter(trainer_tokenizer_kwargs), 'none')}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=default_data_collator,
    callbacks=[CudaMemoryCallback()],
    **trainer_tokenizer_kwargs,
)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print(f'Saved base model to: {OUTPUT_DIR}')
print(f'Training chunks: {len(train_dataset)}')
print(f'Validation chunks: {len(validation_dataset)}')
if torch.cuda.is_available():
    subprocess.run(['nvidia-smi'], check=False)

## 7. Download Base Model

Extract the downloaded zip locally as `model/base_model/`, then run `python model/train_model.py` for the rewrite stage.

In [ ]:
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f'Missing trained base model directory: {OUTPUT_DIR}')

zip_path = Path(shutil.make_archive('/content/base_model', 'zip', OUTPUT_DIR))
print(f'Created: {zip_path}')
files.download(str(zip_path))